In [1]:
import os
import time
import pandas as pd
from ytmusicapi import YTMusic

PLAYLIST_LIMIT=500
PLAYLIST_SONG_LIMIT=10000
yt = YTMusic('../headers_auth.json')

def parse_tracks(track_list):
    tracks = pd.DataFrame(track_list)
    tracks['artistId'] = tracks['artists'].dropna().apply(
        lambda x: x[0]['id'])  # TODO handle > 1 artist
    tracks['artist'] = tracks['artists'].dropna().apply(lambda x: x[0]['name'])
    tracks['albumId'] = tracks['album'].dropna().apply(lambda x: x['id'])
    tracks['album'] = tracks['album'].dropna().apply(lambda x: x['name'])
    tracks = tracks.drop('thumbnails', axis=1)
    tracks = tracks.drop('artists', axis=1)
    return tracks

def parse_playlist(yt, playlist_meta, print_meta=False):
    playlist_meta.pop('thumbnails', None)
    track_list = playlist_meta.pop('tracks', None)
    if print_meta: 
        print(pd.DataFrame.from_dict(playlist_meta, orient='index'))
    tracks = parse_tracks(track_list)
    return tracks, playlist_meta

def create_rating_playlist_subset(tracks, name, rating):
    assert rating in ('LIKE', 'DISLIKE', 'INDIFFERENT')
    filtered_tracks = tracks.loc[tracks['likeStatus'] == rating]
    video_ids = filtered_tracks['videoId'].unique().tolist()
    pl_id = yt.create_playlist(
        title=name + ' ' + rating.lower(), 
        description='generated from %s includes %s subset' % (name, rating),
        privacy_status='PRIVATE', 
        video_ids=video_ids
    )
    print('Created %s playlist with id %s' % (rating, pl_id))

In [2]:
playlists = pd.DataFrame(yt.get_library_playlists(limit=PLAYLIST_LIMIT))
print('Playlists:\n %s' % sorted(playlists.sort_values('title')['title']))


Playlists:
 ['Ambient Psychill', 'Ambient Unrated Albums 2018-2019', 'Beat instrumentals', 'Beats Without Rhymes like', 'Beats indie Chill like', 'Beats indie Chill radio', 'Blues delta roots radio', 'Brass n chill', 'Chillwave', 'Electronic 2010s like', 'Electronic Focus like', 'Electronic Focus radio', 'Electronic House Special like', 'Electronic House Special radio', 'Electronic Innerwaves like', 'Electronic Innerwaves radio', 'Electronic NYC Indie Dance', "Electronic We're Alone Now like", "Electronic We're Alone Now radio", 'Folk like', 'Folk radio', 'Grunge like', 'Grunge radio', 'Hip Hop 1990s like', 'Hip Hop 1990s radio', 'Hip Hop 2000s like', 'Hip Hop 2000s radio', 'Hip Hop Classic West Coast like', 'Hip Hop Classic West Coast radio', 'Hip Hop Hits liked', 'Hip Hop Hits unrated', 'Hip hop It Was a Good Day like', 'Hip hop It Was a Good Day radio', 'Hiphop southeast Ride Around Shining', 'Indie 1990s Rock like', 'Indie 1990s Rock radio', 'Indie 2000s radio', 'Indie Dreams of Fa

In [9]:
# # Example: Create Unrated and Liked Subset Playlist

# playlist_name = 'Analog Grooves'
# playlist = playlists.loc[playlists['title'] == playlist_name].iloc[0] # first match
# metadata = yt.get_playlist(playlist['playlistId'], limit=PLAYLIST_SONG_LIMIT)
# tracks, metadata = parse_playlist(yt, metadata)
# print('Selected Playlist:\n%s' % metadata)

# create_rating_playlist_subset(tracks, playlist_name, 'INDIFFERENT')
# create_rating_playlist_subset(tracks, playlist_name, 'LIKE')

In [10]:
# Example: Group public playlists
public_playlists = {}
privacy = 'PUBLIC'
for i, p in playlists.iterrows():
    if i == 0: continue # skip giant likes playlist
    if 'z_' in p or 'zz_' in p or 'zzz_' in p:
        continue
    metadata = yt.get_playlist(p['playlistId'], limit=PLAYLIST_SONG_LIMIT)
    if metadata['privacy'] == privacy:
        public_playlists[p['playlistId']] = p['title']
        print('Found %s playlist named: %s' % (privacy.lower(), p['title']))

list(public_playlists.values())

[]

In [23]:

#For each playlist, Create Unrated and Liked Subset Playlist, delete original
# 'Jukebox Vintage Party radio', 'Unexpected Best Night Ever', 'punk 1970s British radio', 'Classic Rock Summer', 'nu disco radio', 'beats radio', 'hiphop old school', 'punk 1970s radio', 'Easy-Listening Acid Trip', 'Reggae Dub', 'Post-Punk 1970s-1980s radio', 'Frequencies', 'Monterey Pop Festival 1967', 'Take It Slow', 'Shoegazing', 
playlist_names = ['Blues delta roots radio']

for playlist_name in playlist_names:
    print('Sorting %s in to like and indifferent' % playlist_name)
    playlist = playlists.loc[playlists['title'] == playlist_name].iloc[0] # first match
    metadata = yt.get_playlist(playlist['playlistId'], limit=PLAYLIST_SONG_LIMIT)
    tracks, metadata = parse_playlist(yt, metadata)
    create_rating_playlist_subset(tracks, playlist_name, 'INDIFFERENT')
    time.sleep(20)
    create_rating_playlist_subset(tracks, playlist_name, 'LIKE')
    time.sleep(20)

Sorting Ride Around Shining in to like and indifferent
Created INDIFFERENT playlist with id PLWptjpDqazOxbKdrU6Fuhjn9qDAp2Hrsr
Created LIKE playlist with id PLWptjpDqazOwwlPv5dZ6iQkoHyU5GLCan


In [ ]:
# additional_playlist_names = [
#     'Beats indie Chill indifferent',
#     "Folk radio",
#     "Electronic Focus radio",
#     "Electronic House Special radio",
#     "Electronic We're Alone Now radio",
#     "electronic radio",
#     "future beats radio",
#     "blues radio",
#     'hiphop old radio',
#     'indie loose live chill radio',
#     "Indie Hazy Summer indifferent",
#     'Hip hop It Was a Good Day indifferent',
#     'nu disco radio',
#     'Indie radio',
#     'jazz cool radio',
#     'jazz gloom smooth',
#     'jazz solo guitar radio',
#     'psychedelic classic rock radio',
#     'indie subreddit',
#     'futurebeats subbreddit',
#     'hiphop subreddit',
#     'r.treemusic',
#     'rock classic radio',
#     'rock 1960s classic radio',
#     'rock 1970s classic radio',
#     'Rock 1967-1969 radio',
#     'soul radio',
#     'Soul Classic Sunshine radio'
# ]